In [16]:
import pandas as pd
import numpy as np
import tools


In [17]:
df_kalendarz = pd.read_parquet("dane/interim/fact_inka_records_2023_2026_hard_bez_nigdy_nie_sprzedane.parquet")


In [18]:
df_kalendarz = pd.read_parquet("dane/interim/fact_inka_records_2023_2026_hard_bez_nigdy_nie_sprzedane.parquet")

df_sprzedaz = df_kalendarz[df_kalendarz['TypRuchu'] == 'sprzedaz'].copy()

towid_do_kalendarza = df_sprzedaz['TowId'].unique()
print(f"TowId z jakąkolwiek sprzedażą: {len(towid_do_kalendarza):,}")

data_min = df_kalendarz['Data'].min()
data_max = df_kalendarz['Data'].max()
kalendarz_dni = pd.date_range(start=data_min, end=data_max, freq='D')
print(f"Dni w zakresie: {len(kalendarz_dni):,}")

siatka = pd.MultiIndex.from_product(
    [towid_do_kalendarza, kalendarz_dni], names=['TowId', 'Data']
).to_frame(index=False)
print(f"Rozmiar siatki: {len(siatka):,}")

pelny_kalendarz = siatka.merge(df_sprzedaz, on=['TowId', 'Data'], how='left')
print(f"Rozmiar po scaleniu: {len(pelny_kalendarz):,}")

kolumny_towid = ['NazwaTow', 'EAN', 'AsId', 'Producent', 'NazwaAsort',
                  'NazwaTowCleanName', 'NazwaAsortCleanName', 'JestMartwy']

atrybuty_towid = df_kalendarz[['TowId'] + kolumny_towid].drop_duplicates(subset='TowId')

for kol in kolumny_towid:
    mapa = atrybuty_towid.set_index('TowId')[kol]
    pelny_kalendarz[kol] = pelny_kalendarz[kol].fillna(pelny_kalendarz['TowId'].map(mapa))

pelny_kalendarz['DokId'] = pelny_kalendarz['DokId'].fillna(-1).astype(int)

print(f"\nPuste rekordy (DokId=-1): {(pelny_kalendarz['DokId']==-1).sum():,}")
print(f"Realne rekordy sprzedaży: {(pelny_kalendarz['DokId']!=-1).sum():,}")

TowId z jakąkolwiek sprzedażą: 12,481
Dni w zakresie: 1,127
Rozmiar siatki: 14,066,087
Rozmiar po scaleniu: 15,810,161

Puste rekordy (DokId=-1): 12,436,186
Realne rekordy sprzedaży: 3,373,975


In [19]:
# Kontrola: żadna transakcja nie mogła zginąć ani się zduplikować przy scalaniu
print(f"Wiersze w df_sprzedaz (surowe): {len(df_sprzedaz):,}")
print(f"Realne rekordy w pelny_kalendarz: {(pelny_kalendarz['DokId']!=-1).sum():,}")
print(f"Zgodność: {(pelny_kalendarz['DokId']!=-1).sum() == len(df_sprzedaz)}")

print(f"\nRozmiar: {pelny_kalendarz.shape}")
print(f"Pamięć: {pelny_kalendarz.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"\nKolumny: {pelny_kalendarz.columns.tolist()}")


Wiersze w df_sprzedaz (surowe): 3,373,975
Realne rekordy w pelny_kalendarz: 3,373,975
Zgodność: True

Rozmiar: (15810161, 33)
Pamięć: 6854.5 MB

Kolumny: ['TowId', 'Data', 'DokId', 'Kolejnosc', 'NrPozycji', 'TypPoz', 'IloscPlus', 'IloscMinus', 'CenaPoRab', 'Wartosc', 'KolejnyWDniu', 'NrDok', 'TypDok', 'AktywnyDok', 'Razem', 'DoZaplaty', 'Zaplacono', 'AsId', 'NazwaTow', 'EAN', 'Opis1', 'Producent', 'AktywnyTow', 'NazwaAsort', 'Dokument', 'WplywNaStan', 'MetodaLiczenia', 'Mnoznik', 'TypRuchu', 'CzyNiechciane', 'NazwaTowCleanName', 'NazwaAsortCleanName', 'JestMartwy']


In [20]:
pelny_kalendarz.to_parquet(
    "dane/interim/kalendarz_pelny_towid.parquet",
    compression='zstd',
    index=False
)


In [21]:
# Suma kontrolna
nazwa_pliku = "kalendarz_pelny_towid.parquet"
moj_hash = tools.hash_danych_bezpieczny(f"dane/interim/{nazwa_pliku}")
print(f"Mój hash (posortowane):   {nazwa_pliku}   {moj_hash}")


Mój hash (posortowane):   kalendarz_pelny_towid.parquet   7660340d8f124283973d5f8f6c81de69e2dd96cc37d36e22a37e67bf9808dee6


In [ ]:
# hash pliku kalendarz_pelny_towid.parquet:  7660340d8f124283973d5f8f6c81de69e2dd96cc37d36e22a37e67bf9808dee6